In [20]:
# Run this if facing issue with transformers
# pip install torch torchvision torchaudio transformers

import torch
from datasets import Dataset
from scipy.stats import loguniform
import optuna
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification, pipeline, BertForSequenceClassification, BertTokenizer
import torch

from Utils.import_packages import *

In [44]:
train_df = pd.read_csv('Data/train_labelled.csv')
val_df = pd.read_csv('Data/val_labelled.csv')
test_df = pd.read_csv('Data/test_labelled.csv')

In [45]:
# Apply preprocessing to datasets
train_df["title"] = train_df["title"].str.replace(r'\([A-Za-z]+:[A-Za-z]+\)', '', regex=True)
val_df["title"] = val_df["title"].str.replace(r'\([A-Za-z]+:[A-Za-z]+\)', '', regex=True)
test_df["title"] = test_df["title"].str.replace(r'\([A-Za-z]+:[A-Za-z]+\)', '', regex=True)

train_df["title"] = train_df["title"].apply(lambda text: re.sub(r"\s+", " ", text).strip())
val_df["title"] = val_df["title"].apply(lambda text: re.sub(r"\s+", " ", text).strip())
test_df["title"] = test_df["title"].apply(lambda text: re.sub(r"\s+", " ", text).strip())

train_df = train_df.sort_values(by="date")
val_df = val_df.sort_values(by="date")
test_df = test_df.sort_values(by="date")

In [47]:
# Check distribution of labels
print(train_df['sentiment_label'].value_counts())
print(val_df['sentiment_label'].value_counts())
print(test_df['sentiment_label'].value_counts())

sentiment_label
1    157144
0    151289
Name: count, dtype: int64
sentiment_label
0    17546
1    16441
Name: count, dtype: int64
sentiment_label
0    34213
1    28127
Name: count, dtype: int64


# BERT and FinBERT
BERT and FinBERT are a pre-trained text analysis models, with Finbert particularly trained on financial data. It labels financial data as either "positive", "neutral" or "negative" by assigning a probability to each class, and return the class with the highest probability.
<br> For our use case, we labelled sentiments using only '1' or '0' for the excess 3 day aggregated returns.
<br> For BERT, we will fit the model directly on our test set, since it is pre-trained, just to see how it performs as a baseline compared to FinBERT. This is done by predicting the 3 sentiment probabilities, then using them to map to 1 or 0 based on whether P(positive) >  P(negative)
<BR> For FinBERT, we will first fine tune finBERT to label using 2 classifications on our training set, then using Optuna to search for the best parameters that will return the model with the highest F1 by evaluating its performance on the valuation set. This fine tuned model is then fitted to test set and sentiment labels are predicted again based on the similar mapping logic as before.

## BERT
### Fitting on Test Set

In [122]:
# Load the saved model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3).to(device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model.eval()

# Preprocessing function for BERT
def preprocess_text(text_list):
    return [text.strip().lower() for text in text_list]

# Tokenize data
def preprocess_data(example):
    processed_texts = preprocess_text(example['title'])
    return tokenizer(processed_texts, padding='max_length', truncation=True, max_length=128)

# Tokenize the test set using the saved tokenizer
test_tokenized_df = Dataset.from_pandas(test_df).map(preprocess_data, batched=True)
test_tokenized_df = test_tokenized_df.rename_column("sentiment_label", "labels")
test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Run predictions on the test set
all_logits = []
with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_tokenized_df, batch_size=8):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.softmax(logits, dim=1).cpu().numpy()

prob_negative = probabilities[:, 1]
prob_neutral = probabilities[:, 2]
prob_positive = probabilities[:, 0]

predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

# Save predictions to test DataFrame
test_df_results_bert = test_df.copy()
test_df_results_bert['prob_negative'] = prob_negative
test_df_results_bert['prob_neutral'] = prob_neutral
test_df_results_bert['prob_positive'] = prob_positive
test_df_results_bert['finbert_sentiment_labels'] = predicted_labels

# Evaluate metrics if ground-truth labels exist
if 'sentiment_label' in test_df_results_bert.columns:
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_df_results_bert['sentiment_label'], predicted_labels, average='weighted'
    )
    accuracy = accuracy_score(test_df_results_bert['sentiment_label'], predicted_labels)

    print(f"Test Accuracy: {accuracy}")
    print(f"Test Precision: {precision}")
    print(f"Test Recall: {recall}")
    print(f"Test F1-Score: {f1}")

test_df_results_bert

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/21491 [00:00<?, ? examples/s]

Test Accuracy: 0.480107952166023
Test Precision: 0.5515600853585115
Test Recall: 0.480107952166023
Test F1-Score: 0.31295279109530244


,title,source,date,sentiment_label,prob_negative,prob_neutral,prob_positive,finbert_sentiment_labels
14637,NiSource Inc. Completes NIPSCO Minority Equity...,https://news.google.com/rss/articles/CBMiwgFBV...,2024-01-02,1,0.291833,0.209391,0.498776,1
3380,Carrier Completes Acquisition of Viessmann Cli...,https://news.google.com/rss/articles/CBMiugFBV...,2024-01-02,0,0.283739,0.197097,0.519164,1
3581,CBRE Acquires Full Ownership of CBRE Raleigh -...,https://news.google.com/rss/articles/CBMiuAFBV...,2024-01-02,0,0.278122,0.199393,0.522485,1
4432,Clorox partners with Bravo & NBC Universal … b...,https://news.google.com/rss/articles/CBMitwFBV...,2024-01-03,0,0.305076,0.275804,0.419120,1
2930,"Brown & Brown, Inc. Announces 2023 Fourth-Quar...",https://news.google.com/rss/articles/CBMivwFBV...,2024-01-03,0,0.290323,0.225659,0.484017,1
...,...,...,...,...,...,...,...,...
10567,HSBC downgrades HP Inc to 'hold' on cost press...,https://news.google.com/rss/articles/CBMiwgFBV...,2024-11-29,1,0.276897,0.208160,0.514943,1
1162,Amcor & Kolon to Develop Sustainable Polyester...,https://news.google.com/rss/articles/CBMikAFBV...,2024-11-29,0,0.303424,0.242138,0.454439,1
10566,HP Inc. Shares Acquired by Glenmede Trust Co. ...,https://news.google.com/rss/articles/CBMirgFBV...,2024-11-29,1,0.288788,0.192410,0.518802,1
2642,Healthcare of Ontario Pension Plan Trust Fund ...,https://news.google.com/rss/articles/CBMi4gFBV...,2024-11-29,0,0.295446,0.229558,0.474996,1


In [124]:
test_df_results_bert.to_csv('result/test_df_results_bert.csv', index=False)

## Prosus AI/FinBERT
### Optuna Hyperparameter Optimisation

In [48]:
train_subset_size = int(len(train_df) * 0.05) # take 5% of train set

# Sample proportionally across all dates
train_subset_df = train_df.iloc[::len(train_df) // train_subset_size]
train_subset_df = train_subset_df.sort_values(by=['date']).reset_index(drop=True)

In [50]:
# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# Tokenize data
def preprocess_data(example):
    return tokenizer(example['title'], padding='max_length', max_length=128, truncation=True)

train_subset_tokenized_df = Dataset.from_pandas(train_subset_df).map(preprocess_data, batched=True)
train_subset_tokenized_df = train_subset_tokenized_df.rename_column("sentiment_label", "labels")
train_subset_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

val_tokenized_df = Dataset.from_pandas(val_df).map(preprocess_data, batched=True)
val_tokenized_df = val_tokenized_df.rename_column("sentiment_label", "labels")
val_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Define metrics for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

    prob_negative = probabilities[:, 1]
    prob_neutral = probabilities[:, 2]
    prob_positive = probabilities[:, 0]

    predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predicted_labels, average='weighted')
    accuracy = accuracy_score(labels, predicted_labels)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Define the Optuna objective function
def objective(trial):
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-5)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    num_train_epochs = trial.suggest_int('num_train_epochs', 2, 5)

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=3).to(device)

    # Define training arguments
    training_args = TrainingArguments(
        output_dir="./train_subset_results",
        eval_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_dir="./train_subset_logs",
        logging_steps=10,
        disable_tqdm=False,
        load_best_model_at_end=True
    )

    # Define the Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset_tokenized_df,
        eval_dataset=val_tokenized_df,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Train model
    trainer.train()

    # Evaluate model
    eval_results = trainer.evaluate()
    return eval_results['eval_f1']

# Initialize an Optuna study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Output the best hyperparameters
print("Best hyperparameters:")
print(study.best_params)

Using device: cuda


Map:   0%|          | 0/15422 [00:00<?, ? examples/s]

Map:   0%|          | 0/33987 [00:00<?, ? examples/s]

[I 2024-12-17 02:14:33,442] A new study created in memory with name: no-name-2a9ab710-68a5-49db-a681-26318b002937
C:\Users\Tylus\AppData\Local\Temp\ipykernel_3608\1610795840.py:46: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-5)
C:\Users\Tylus\AppData\Local\Temp\ipykernel_3608\1610795840.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.687100,0.699016,0.513755,0.445711,0.513755,0.356724
2,0.702200,0.697866,0.500485,0.487091,0.500485,0.453472


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.697300,0.704087,0.514167,0.449709,0.514167,0.356362
2,0.704600,0.694375,0.495042,0.496897,0.495042,0.494289
3,0.678900,0.697420,0.501309,0.492790,0.501309,0.473225


[I 2024-12-17 05:59:10,128] Trial 48 finished with value: 0.4942893523208349 and parameters: {'learning_rate': 1.6015129108835143e-05, 'batch_size': 32, 'num_train_epochs': 3}. Best is trial 40 with value: 0.4965826996424298.
C:\Users\Tylus\AppData\Local\Temp\ipykernel_3608\1610795840.py:46: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-5)
C:\Users\Tylus\AppData\Local\Temp\ipykernel_3608\1610795840.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.697100,0.702784,0.516256,0.508393,0.516256,0.351603
2,0.703200,0.695653,0.500250,0.494034,0.500250,0.482995
3,0.683700,0.696277,0.499397,0.492253,0.499397,0.478831


[I 2024-12-17 06:03:17,787] Trial 49 finished with value: 0.48299512564764446 and parameters: {'learning_rate': 1.814886178301257e-05, 'batch_size': 32, 'num_train_epochs': 3}. Best is trial 40 with value: 0.4965826996424298.


Best hyperparameters:
{'learning_rate': 1.3338568413149221e-05, 'batch_size': 32, 'num_train_epochs': 3}


### Fine Tuning on Full Train Set

In [51]:
# Tokenize data
train_tokenized_df = Dataset.from_pandas(train_df).map(preprocess_data, batched=True)
train_tokenized_df = train_tokenized_df.rename_column("sentiment_label", "labels")
train_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

val_tokenized_df = Dataset.from_pandas(val_df).map(preprocess_data, batched=True)
val_tokenized_df = val_tokenized_df.rename_column("sentiment_label", "labels")
val_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Train and evaluate the model using the best hyperparameters from Optuna
best_params = study.best_params

model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert', num_labels=3).to(device)

training_args = TrainingArguments(
    output_dir="./train_results",
    evaluation_strategy="epoch",
    learning_rate=best_params['learning_rate'],
    per_device_train_batch_size=best_params['batch_size'],
    num_train_epochs=best_params['num_train_epochs'],
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./train_logs",
    logging_steps=10,
    disable_tqdm=False,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_df,
    eval_dataset=val_tokenized_df,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Save trained model
output_dir = "./finbert_finetuned"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Final evaluation on the validation set
model.eval()
all_logits = []

with torch.no_grad():
    for batch in torch.utils.data.DataLoader(val_tokenized_df, batch_size=best_params['batch_size']):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.softmax(logits, dim=1).cpu().numpy()

# Extract probabilities for each class
prob_negative = probabilities[:, 1]
prob_neutral = probabilities[:, 2]
prob_positive = probabilities[:, 0]

# Transform probabilities into labels
predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

# Add probabilities and predictions to the validation DataFrame
val_df['prob_negative'] = prob_negative
val_df['prob_neutral'] = prob_neutral
val_df['prob_positive'] = prob_positive
val_df['finbert_sentiment_labels'] = predicted_labels

# Final evaluation metrics
if 'sentiment_label' in val_df.columns:
    precision, recall, f1, _ = precision_recall_fscore_support(val_df['sentiment_label'], predicted_labels, average='weighted')
    accuracy = accuracy_score(val_df['sentiment_label'], predicted_labels)

    print(f"Validation Accuracy: {accuracy}")
    print(f"Validation Precision: {precision}")
    print(f"Validation Recall: {recall}")
    print(f"Validation F1-Score: {f1}")

val_df

Map:   0%|          | 0/308433 [00:00<?, ? examples/s]

Map:   0%|          | 0/33987 [00:00<?, ? examples/s]

C:\Users\Tylus\PycharmProjects\QF634_Research_Methods\.venv2\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Tylus\AppData\Local\Temp\ipykernel_3608\3609084924.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.694900,0.693909,0.493630,0.486311,0.493630,0.475570
2,0.686000,0.700325,0.497808,0.491139,0.497808,0.480052
3,0.672600,0.711155,0.498573,0.492991,0.498573,0.484602


Validation Accuracy: 0.49362991732132877
Validation Precision: 0.4863109320370542
Validation Recall: 0.49362991732132877
Validation F1-Score: 0.47557032668909943


,title,source,topic,company name(s) - cleaned,date,sentiment_label,prob_negative,prob_neutral,prob_positive,finbert_sentiment_labels
33320,W. R. Berkley Corporation Declares Special Cas...,Business Wire,Property and Casualty Insurance,W. R. Berkley Corporation,2023-01-03,0,0.524493,0.000056,0.475451,0
7455,"FLEETCOR Technologies, Inc. completed the acqu...",Capital IQ Transaction Database,"Corpay, Inc. (NYSE:CPAY) (Transaction and Paym...","Corpay, Inc.",2023-01-03,0,0.488968,0.000058,0.510974,1
2387,"ANSYS, Inc. acquired Rocky DEM, S.L from Engin...",Capital IQ Transaction Database,"ANSYS, Inc. (NasdaqGS:ANSS) (Application Softw...","ANSYS, Inc.",2023-01-03,0,0.500393,0.000061,0.499546,0
30048,Thermo Fisher Scientific Inc. completed the ac...,Capital IQ Transaction Database,Five Arrows Managers SAS (Asset Management and...,Five Arrows Managers SAS; Nordic Capital; The ...,2023-01-03,1,0.506100,0.000071,0.493829,0
20828,"MetLife, Inc. has completed a Fixed-Income Off...",Capital IQ Transaction Database,Life and Health Insurance,"MetLife, Inc.",2023-01-03,1,0.529121,0.000096,0.470783,0
...,...,...,...,...,...,...,...,...,...,...
94,"Agilent Technologies, Inc., $ 0.236, Cash Divi...",Financial Times,Life Sciences Tools and Services,"Agilent Technologies, Inc.",2023-12-29,0,0.529555,0.000053,0.470392,0
22909,"Micron Technology, Inc., $ 0.115, Cash Dividen...",Financial Times,Semiconductors,"Micron Technology, Inc.",2023-12-29,0,0.536853,0.000053,0.463094,0
32161,"Ventas, Inc., $ 0.45, Cash Dividend, Dec-29-2023",Financial Times,Health Care REITs,"Ventas, Inc.",2023-12-29,0,0.528051,0.000059,0.471891,0
11381,"Essex Property Trust, Inc., $ 2.31, Cash Divid...",Financial Times,Multi-Family Residential REITs,"Essex Property Trust, Inc.",2023-12-29,0,0.510617,0.000059,0.489324,0


### Fitting on Test Set

In [52]:
# Load the saved model and tokenizer
output_dir = "./finbert_finetuned"
model = AutoModelForSequenceClassification.from_pretrained(output_dir).to(device)
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model.eval()

# Tokenize the test set using the saved tokenizer
test_tokenized_df = Dataset.from_pandas(test_df).map(preprocess_data, batched=True)
test_tokenized_df = test_tokenized_df.rename_column("sentiment_label", "labels")
test_tokenized_df.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Run predictions on the test set
all_logits = []
with torch.no_grad():
    for batch in torch.utils.data.DataLoader(test_tokenized_df, batch_size=best_params['batch_size']):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        all_logits.append(logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.softmax(logits, dim=1).cpu().numpy()

prob_negative = probabilities[:, 1]
prob_neutral = probabilities[:, 2]
prob_positive = probabilities[:, 0]

predicted_labels = [
    1 if prob_positive > max(prob_negative, prob_neutral) else 0
    for prob_negative, prob_neutral, prob_positive in zip(prob_negative, prob_neutral, prob_positive)
]

# Save predictions to test DataFrame
test_df_results = test_df.copy()
test_df_results['prob_negative'] = prob_negative
test_df_results['prob_neutral'] = prob_neutral
test_df_results['prob_positive'] = prob_positive
test_df_results['finbert_sentiment_labels'] = predicted_labels

# Evaluate metrics if ground-truth labels exist
if 'sentiment_label' in test_df_results.columns:
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_df_results['sentiment_label'], predicted_labels, average='weighted'
    )
    accuracy = accuracy_score(test_df_results['sentiment_label'], predicted_labels)

    print(f"Test Accuracy: {accuracy}")
    print(f"Test Precision: {precision}")
    print(f"Test Recall: {recall}")
    print(f"Test F1-Score: {f1}")

test_df_results

Map:   0%|          | 0/62340 [00:00<?, ? examples/s]

Test Accuracy: 0.4972409367982034
Test Precision: 0.4935980427519241
Test Recall: 0.4972409367982034
Test F1-Score: 0.49478718051870496


,title,source,topic,company name(s) - cleaned,date,sentiment_label,prob_negative,prob_neutral,prob_positive,finbert_sentiment_labels
7597,"Franklin Resources, Inc., $ 0.31, Cash Dividen...",Financial Times,Asset Management and Custody Banks,"Franklin Resources, Inc.",2024-01-02,0,0.517337,0.000066,0.482597,0
34121,KKR & Co. Inc. completed the acquisition of re...,Capital IQ Transaction Database,KKR & Co. Inc. (NYSE:KKR) (Asset Management an...,KKR & Co. Inc.,2024-01-02,0,0.498220,0.000100,0.501680,1
52817,Kimco Realty Corporation completed the acquisi...,Capital IQ Transaction Database,"BlackRock, Inc. (NYSE:BLK) (Asset Management a...","BlackRock, Inc.",2024-01-02,1,0.505592,0.000067,0.494341,0
8924,"Brown & Brown, Inc. completed the acquisition ...",Capital IQ Transaction Database,"Assets of Caton Hosey Insurance, Inc. (Insuran...","Assets of Caton Hosey Insurance, Inc.; Brown &...",2024-01-02,1,0.503192,0.000079,0.496729,0
8493,European Medicines Agency Validates Bristol Br...,Business Wire,Pharmaceuticals,Bristol-Myers Squibb Company,2024-01-02,1,0.493889,0.000056,0.506055,1
...,...,...,...,...,...,...,...,...,...,...
17395,Is Dell Technologies (DELL) the Best Computer ...,https://news.google.com/rss/articles/CBMiigFBV...,NaN,NaN,2024-12-13,0,0.496004,0.000058,0.503938,1
17396,"M&T Bank Corp Purchases 9,630 Shares of Dell T...",https://news.google.com/rss/articles/CBMivgFBV...,NaN,NaN,2024-12-13,0,0.496931,0.000061,0.503008,1
17397,Dell Technologies Inc. Shares Acquired by Toro...,https://news.google.com/rss/articles/CBMiwwFBV...,NaN,NaN,2024-12-13,0,0.485511,0.000068,0.514421,1
56132,TVLine Items The Day of the Jackal on NBC Dinn...,https://tvline.com/news/the-day-of-the-jackal-...,entertainment,NaN,2024-12-13,0,0.492301,0.000075,0.507624,1


In [54]:
# Save predictions to the test dataframe
test_df_results.to_csv("results/test_df_results.csv", index=False)